In [1]:
!pip install tensorflow opencv-python mediapipe scikit-learn matplotlib
# Verify installations
import tensorflow as tf
import cv2 as cv2
import mediapipe as mp
from sklearn import datasets
import matplotlib.pyplot as plt
import time
import os
import numpy as np
print("All libraries imported successfully.")
# NOTE: always run this when freshly opening this/working on it

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 615.5/615.5 MB 32.5 MB/s eta 0:00:0000:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 63.0/63.0 MB 51.4 MB/s eta 0:00:00:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 35.6/35.6 MB 88.8 MB/s eta 0:00:00:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 5.9/5.9 MB 88.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 5.4/5.4 MB 95.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.3/1.3 MB 75.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 24.5/24.5 MB 92.6 MB/s eta 0:00:00:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.2/2.2 MB 82.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 18.0/18.0 MB 95.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 5.5/5.5 MB 80.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.3/2.3 MB 83.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 102.0/102.0 MB 76.2 MB/s eta 0:00:0000:0100:01
   ━━━━━━━━━━━━━━━━━━━

2025-02-15 02:38:50.780632: I tensorflow/core/util/port.cc:153] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.
2025-02-15 02:38:50.797984: I external/local_xla/xla/tsl/cuda/cudart_stub.cc:32] Could not find cuda drivers on your machine, GPU will not be used.
2025-02-15 02:38:50.982264: I external/local_xla/xla/tsl/cuda/cudart_stub.cc:32] Could not find cuda drivers on your machine, GPU will not be used.
2025-02-15 02:38:51.072368: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:477] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1739587131.193934   13303 cuda_dnn.cc:8310] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1739587131.22

ImportError: libGL.so.1: cannot open shared object file: No such file or directory

In [160]:
mp_holistic = mp.solutions.holistic # Holistic models
mp_drawing = mp.solutions.drawing_utils #drawing utils
print("holistics drawing is ready") # Confirms it

holistics drawing is ready


In [162]:
def mediapipe_detection(image, model):
    image = cv2.cvtColor(image, cv2.COLOR_BGR2RGB) #From BGR (Blue, Green, Red) to RGB
    image.flags.writeable = False #Restricts the image so it wouldnt be modified
    results = model.process(image) # Prediciton
    image.flags.writeable = True #Allows image to be modified
    image = cv2.cvtColor(image, cv2.COLOR_RGB2BGR) #From RGB to BGR (Blue, Green, Red)
    return image, results

In [164]:
def draw_landmarks(image, results):
    mp_drawing.draw_landmarks(image, results.face_landmarks, mp_holistic.FACEMESH_TESSELATION) # this is face connections replaced by attribute "facemesh"
    mp_drawing.draw_landmarks(image, results.pose_landmarks, mp_holistic.POSE_CONNECTIONS) # draws pose connections
    mp_drawing.draw_landmarks(image, results.left_hand_landmarks, mp_holistic.HAND_CONNECTIONS) # draws left hand 
    mp_drawing.draw_landmarks(image, results.right_hand_landmarks, mp_holistic.HAND_CONNECTIONS)# draws right hand

In [166]:
def draw_styled_landmarks(image, results):
    mp_drawing.draw_landmarks(image, results.face_landmarks, mp_holistic.FACEMESH_TESSELATION,
        mp_drawing.DrawingSpec(color=(80, 110, 10), thickness=1, circle_radius=1),  # the landmark aka circle color
        mp_drawing.DrawingSpec(color=(80, 0, 121), thickness=1, circle_radius=1)  # the connection aka line color
    )  # this is face connections replaced by attribute "facemesh"

    mp_drawing.draw_landmarks(image, results.pose_landmarks, mp_holistic.POSE_CONNECTIONS,
        mp_drawing.DrawingSpec(color=(80, 22, 10), thickness=2, circle_radius=4),
        mp_drawing.DrawingSpec(color=(80, 44, 121), thickness=2, circle_radius=2)
    )  # draws pose connections

    mp_drawing.draw_landmarks(image, results.left_hand_landmarks, mp_holistic.HAND_CONNECTIONS,
        mp_drawing.DrawingSpec(color=(121, 22, 76), thickness=2, circle_radius=4),
        mp_drawing.DrawingSpec(color=(121, 44, 250), thickness=2, circle_radius=2)
    )  # draws left hand 

    mp_drawing.draw_landmarks(image, results.right_hand_landmarks, mp_holistic.HAND_CONNECTIONS,
        mp_drawing.DrawingSpec(color=(80, 110, 10), thickness=2, circle_radius=4),
        mp_drawing.DrawingSpec(color=(80, 0, 121), thickness=2, circle_radius=2)
    )  # draws right hand NOTE: FIX THIS ISSUE WITH COLOR CUSTOMISATION LATER!!!!!!!(fixed)


In [170]:
import cv2

cap = cv2.VideoCapture(0)
# Set mediapipe model 
with mp_holistic.Holistic(min_detection_confidence=0.5, min_tracking_confidence=0.5) as holistic:
    while cap.isOpened():

        # Read feed
        ret, frame = cap.read()

        # Make detections
        image, results = mediapipe_detection(frame, holistic)
        print(results)
        
        # draws landsmarks(key points)
        draw_styled_landmarks(image, results)
        
        # Show to screen
        cv2.imshow('Mizo Frame', image)

        # Break gracefully
        if cv2.waitKey(10) & 0xFF == ord('x'):
            break
    cap.release()
    cv2.destroyAllWindows()

<class 'mediapipe.python.solution_base.SolutionOutputs'>
<class 'mediapipe.python.solution_base.SolutionOutputs'>
<class 'mediapipe.python.solution_base.SolutionOutputs'>
<class 'mediapipe.python.solution_base.SolutionOutputs'>
<class 'mediapipe.python.solution_base.SolutionOutputs'>
<class 'mediapipe.python.solution_base.SolutionOutputs'>
<class 'mediapipe.python.solution_base.SolutionOutputs'>
<class 'mediapipe.python.solution_base.SolutionOutputs'>
<class 'mediapipe.python.solution_base.SolutionOutputs'>
<class 'mediapipe.python.solution_base.SolutionOutputs'>
<class 'mediapipe.python.solution_base.SolutionOutputs'>
<class 'mediapipe.python.solution_base.SolutionOutputs'>
<class 'mediapipe.python.solution_base.SolutionOutputs'>
<class 'mediapipe.python.solution_base.SolutionOutputs'>
<class 'mediapipe.python.solution_base.SolutionOutputs'>
<class 'mediapipe.python.solution_base.SolutionOutputs'>
<class 'mediapipe.python.solution_base.SolutionOutputs'>
<class 'mediapipe.python.soluti

In [26]:
len(results.left_hand_landmarks.landmark) #gets results here in this case finds the amount of left hand

AttributeError: 'NoneType' object has no attribute 'landmark'

In [22]:
results

mediapipe.python.solution_base.SolutionOutputs

In [33]:
draw_landmarks(frame, results)

In [ ]:
plt.imshow(cv2.cvtColor(frame, cv2.COLOR_BGR2RGB))  # note this changes it from bgr to rgb cuz it looks funky :)

In [46]:
pose = [] 
for res in results.pose_landmarks.landmark:
    test = np.array([res.x, res.y, res.z, res.visibility])
    pose.append(test)

In [48]:
pose = np.array([[res.x, res.y, res.z, res.visibility] for res in results.pose_landmarks.landmark]).flatten() if results.pose_landmarks else np.zeros(132)
face = np.array([[res.x, res.y, res.z] for res in results.face_landmarks.landmark]).flatten() if results.face_landmarks else np.zeros(1404) # error if statement in case the face is not in frame
lh = np.array([[res.x, res.y, res.z] for res in results.left_hand_landmarks.landmark]).flatten() if results.left_hand_landmarks else np.zeros(21*3) # error if statement in case the left hand is not in frame
rh = np.array([[res.x, res.y, res.z] for res in results.right_hand_landmarks.landmark]).flatten() if results.right_hand_landmarks else np.zeros(21*3)# error if statement in case the right hand is not in frame

In [50]:
def extract_keypoints(results):
    pose = np.array([[res.x, res.y, res.z, res.visibility] for res in results.pose_landmarks.landmark]).flatten() if results.pose_landmarks else np.zeros(132)
    face = np.array([[res.x, res.y, res.z] for res in results.face_landmarks.landmark]).flatten() if results.face_landmarks else np.zeros(1404) # error if statement in case the face is not in frame
    lh = np.array([[res.x, res.y, res.z] for res in results.left_hand_landmarks.landmark]).flatten() if results.left_hand_landmarks else np.zeros(21*3) # error if statement in case the left hand is not in frame
    rh = np.array([[res.x, res.y, res.z] for res in results.right_hand_landmarks.landmark]).flatten() if results.right_hand_landmarks else np.zeros(21*3)# error if statement in case the right hand is not in frame
    return np.concatenate([pose, face, lh, rh])# merges all the points together

In [52]:
# where all the data from 3. goes
DATA_PATH = os.path.join("MP_Data")
 
# actions to try to detect/recognise 
actions = np.array(["1","2","3"]) # NOTE: ADD more to make the ai more better and useful 
# 30 videos worth of data NOTE: you can make it bigger for more accuracy
no_sequences = 60
# 30 frames per video NOTE: you can change it to 60
sequence_length = 60

In [54]:
for action in actions:
    for sequence in range(no_sequences):
        try:
            os.makedirs(os.path.join(DATA_PATH, action, str(sequence)))
        except:
            pass

In [47]:
import cv2

cap = cv2.VideoCapture(0)

# Set mediapipe model 
with mp_holistic.Holistic(min_detection_confidence=0.5, min_tracking_confidence=0.5) as holistic:

    # Loops through the actions
    for action in actions:
        # Loops through no_sequences
        for sequence in range(no_sequences):
            # Loops through sequence_length
            for frame_num in range(sequence_length):
                
                # Read feed
                ret, frame = cap.read()
        
                # Make detections
                image, results = mediapipe_detection(frame, holistic)
                print(results)
                
                # Draw landmarks (key points)
                draw_styled_landmarks(image, results)

                # Applying Collection delay to collect action easier
                if frame_num == 0:
                    cv2.putText(image, "STARTING COLLECTION", (120, 200),
                                cv2.FONT_HERSHEY_SIMPLEX, 1, (0, 255, 0), 4, cv2.LINE_AA)
                    cv2.putText(image, "Collecting frames for {} Video Number {}".format(action, sequence), (15, 12),
                                cv2.FONT_HERSHEY_SIMPLEX, 0.5, (0, 0, 255), 1, cv2.LINE_AA)
                    cv2.waitKey(3000)
                else:
                    cv2.putText(image, "Collecting frames for {} Video Number {}".format(action, sequence), (15, 12),
                                cv2.FONT_HERSHEY_SIMPLEX, 0.5, (0, 0, 255), 1, cv2.LINE_AA)
                    # Show to screen
                    cv2.imshow('Mizo Frame', image)

                # Exports keypoints to file directory
                keypoints = extract_keypoints(results)
                npy_path = os.path.join(DATA_PATH, action, str(sequence), str(frame_num))
                np.save(npy_path, keypoints)

                # Break gracefully
                if cv2.waitKey(10) & 0xFF == ord('x'):
                    break
                     
    cap.release()
    cv2.destroyAllWindows()


<class 'mediapipe.python.solution_base.SolutionOutputs'>
<class 'mediapipe.python.solution_base.SolutionOutputs'>
<class 'mediapipe.python.solution_base.SolutionOutputs'>
<class 'mediapipe.python.solution_base.SolutionOutputs'>
<class 'mediapipe.python.solution_base.SolutionOutputs'>
<class 'mediapipe.python.solution_base.SolutionOutputs'>
<class 'mediapipe.python.solution_base.SolutionOutputs'>
<class 'mediapipe.python.solution_base.SolutionOutputs'>
<class 'mediapipe.python.solution_base.SolutionOutputs'>
<class 'mediapipe.python.solution_base.SolutionOutputs'>
<class 'mediapipe.python.solution_base.SolutionOutputs'>
<class 'mediapipe.python.solution_base.SolutionOutputs'>
<class 'mediapipe.python.solution_base.SolutionOutputs'>
<class 'mediapipe.python.solution_base.SolutionOutputs'>
<class 'mediapipe.python.solution_base.SolutionOutputs'>
<class 'mediapipe.python.solution_base.SolutionOutputs'>
<class 'mediapipe.python.solution_base.SolutionOutputs'>
<class 'mediapipe.python.soluti

In [56]:
label_map = {label:num for num, label in enumerate(actions)}

In [95]:
sequences, label =[], []
for action in actions:
    for sequence in range(no_sequences):
        window = []
        for frame_num in range(sequence_length):
            res = np.load(os.path.join(DATA_PATH, action , str(sequence), "{}.npy".format(frame_num)))
            window.append(res)
        sequences.append(window)
        label.append(label_map[action])

In [97]:
X= np.array(sequences)

In [99]:
from tensorflow.keras.utils import to_categorical

y = to_categorical(label).astype(int)

In [101]:
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.05) # Training and testing partition

In [180]:
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import LSTM, Dense, Dropout
from tensorflow.keras.callbacks import TensorBoard
print("LSTM imported")

LSTM imported


In [182]:
from tensorflow.keras.callbacks import TensorBoard

# Define the path for the "Logs" folder
log_dir = os.path.join("Logs")

# Ensure that the "Logs" folder exists
if not os.path.exists(log_dir):
    os.makedirs(log_dir)

# Initialize the TensorBoard callback with the log directory
tb_callback = TensorBoard(log_dir=log_dir)

# You can now use tb_callback in model training

In [184]:
model = Sequential()
model.add(LSTM(64, return_sequences=True, activation="tanh", input_shape=(30,1662)))
model.add(Dropout(0.5))
model.add(LSTM(128, return_sequences=True, activation="tanh"))
model.add(Dropout(0.5))
model.add(LSTM(64, return_sequences=False, activation="tanh"))
model.add(Dense(64, activation="relu"))
model.add(Dropout(0.5))
model.add(Dense(32, activation="relu"))
model.add(Dense(actions.shape[0], activation="softmax"))

AttributeError: 'list' object has no attribute 'shape'

In [109]:
model.compile(optimizer="Adam", loss="categorical_crossentropy", metrics=["categorical_accuracy"])  #metrics is the variable that concludes the model accureate to what it is displaying

In [ ]:
model.fit(X_train, y_train, epochs=2000, callbacks=[tb_callback]) #NOTE use tensorboard web to see data through cmd "tensorboard --logdir=logs/train"

8.Making Predictions

In [ ]:
model.predict(X_test)

In [ ]:
actions[np.arg.max(res[0])]

In [ ]:
actions[np.arg.max(y_test[0])]

9.Saving the Data(Weights)

In [69]:
model.save("action.h5")

In [ ]:
del model #NOTE: USE IT IF MODEL IS UNSTABLE AND/OR UNUSABLE

In [ ]:
model.load_weights("action.h5") # Recovers the file if user accidently deletes it

10.Evaluation using Confusion Matrix and Accuracy

In [71]:
from sklearn.metrics import multilabel_confusion_matrix, accuracy_score

In [82]:
yhat = model.predict(X_test)

1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 242ms/step


In [83]:
ytrue = np.argmax(y_test, axis=1).tolist()
yhat = np.argmax(yhat, axis=1).tolist()

In [84]:
multilabel_confusion_matrix(ytrue, yhat) #NOTE refer to the confusion matrix with false negatives and true positives

NameError: name 'multilabel_confusion_matrix' is not defined

In [ ]:
accuracy_score(ytrue, yhat)

11. TESTING IN REAL TIME 

In [156]:
colors = [(245,117,16), (117,245,16), (16,117,245)]
def prob_viz(res, actions, input_frame, colors):
    output_frame = input_frame.copy()
    for num, prob in enumerate(res):
        cv2.rectangle(output_frame, (0,60+num*40), (int(prob*100), 90+num*40), colors[num], -1)
        cv2.putText(output_frame, actions[num], (0, 85+num*40), cv2.FONT_HERSHEY_SIMPLEX, 1, (255,255,255), 2, cv2.LINE_AA)

    return output_frame

In [2]:
# New detections
sequence = []  # collects the 30 frames
sentence = [] 
predictions = []  # predicts the action during transition
threshold = 0.5  # optimize the threshold for accuracy

cap = cv2.VideoCapture(0)
# Set mediapipe model 
with mp_holistic.Holistic(min_detection_confidence=0.5, min_tracking_confidence=0.5) as holistic:
    while cap.isOpened():

        # Read feed
        ret, frame = cap.read()
        
        if not ret:
            print("Failed to grab frame")
            break

        # Make detections
        image, results = mediapipe_detection(frame, holistic)
        
        # Check if the results contain landmarks
        if not results.pose_landmarks:
            print("No landmarks detected")
            continue

        print(results)
        
        # Draw landmarks (key points)
        draw_styled_landmarks(image, results)

        # Prediction logic
        keypoints = extract_keypoints(results)
        sequence.append(keypoints)
        sequence = sequence[-30:]

        if len(sequence) == 30:
            res = model.predict(np.expand_dims(sequence, axis=0))[0]
            print(f"Raw model prediction: {res}")
            
            if res.size > 0:
                predicted_action = actions[np.argmax(res)]
                print(f"Predicted action: {predicted_action}")
                
                # Visualization logic
                if len(predictions) > 0 and np.unique(predictions[-10:])[0] == np.argmax(res):
                    if res[np.argmax(res)] > threshold:
                        
                        if len(sentence) > 0:
                            if predicted_action != sentence[-1]:
                                sentence.append(predicted_action)
                        else:
                            sentence.append(predicted_action)

                sentence = sentence[-5:]  # Keep only the last 5 predictions
                print(f"Current sentence: {sentence}")
                
                image = prob_viz(res, actions, image, colors)

                cv2.rectangle(image, (0, 0), (640, 40), (245, 117, 16), -1)
                cv2.putText(image, " ".join(sentence), (3, 30),
                            cv2.FONT_HERSHEY_SIMPLEX, 1, (255, 255, 255), 2, cv2.LINE_AA)  # Determines the box on the top left
        
        # Show to screen
        cv2.imshow('Mizo Frame', image)

        # Break gracefully
        if cv2.waitKey(10) & 0xFF == ord('x'):
            break

cap.release()
cv2.destroyAllWindows()


NameError: name 'cv2' is not defined